# Awaaz Kharcha — multilingual NLP fine-tuning (Colab)

One-click reproduction of the `nlp/` experiments. Runtime → **GPU** recommended.
Tasks: category classification, NER (AMOUNT/ITEM/PAYEE/DATE), language ID.
Models compared via `--model`: `ai4bharat/IndicBERT_v2`, `google/muril-base-cased` (+`--lora`), `xlm-roberta-base`, `distilbert-base-multilingual-cased`.

In [ ]:
# clone repo (or upload nlp/ folder if running standalone)
!git clone https://github.com/PoornaSri26/Awaaz-Kharcha.git
%cd Awaaz-Kharcha
!pip install -q -r requirements-dev.txt
!pip install -q transformers datasets seqeval peft

In [ ]:
# build / inspect the dataset
!python nlp/data/gen_synthetic.py --per-lang 60
!head -3 nlp/data/synthetic.jsonl

In [ ]:
# quick CPU smoke test (optional) — full runs below use GPU
!python nlp/train_classifier.py --model distilbert-base-multilingual-cased \
    --data nlp/data/synthetic.jsonl --out nlp/models/category_distilbert \
    --epochs 3 --max-steps 100

In [ ]:
# full experiments — IndicBERT full FT, MuRIL LoRA, XLM-R langid
!python nlp/train_classifier.py --model ai4bharat/IndicBERT_v2 \
    --out nlp/models/category_indicbert --epochs 4
!python nlp/train_classifier.py --model google/muril-base-cased --lora \
    --out nlp/models/category_muril_lora --epochs 4 --lr 1e-4
!python nlp/train_ner.py --model ai4bharat/IndicBERT_v2 --out nlp/models/ner --epochs 4
!python nlp/train_langid.py --model xlm-roberta-base --out nlp/models/langid_xlmr --epochs 4

In [ ]:
# aggregate results
!python nlp/evaluate.py

In [ ]:
# serve best classifier through the existing app (optional)
# !python -c "from transformers import pipeline; p=pipeline('text-classification', model='nlp/models/category_indicbert/final'); print(p('aaj chai 20'))"